In [ ]:
from data_prep import *

import os
import pandas as pd

from sklearn.model_selection import train_test_split

In [ ]:
data_abs_path = r"C:\Users\jason\Desktop\Storm Damage Analsis - Storm Level\data_wrangling\merged_data"

parent_dat = None
for folder in os.listdir(data_abs_path):
    for file in os.listdir(os.path.join(data_abs_path, folder)):
        if file.endswith('.csv'):
            dat = pd.read_csv(os.path.join(data_abs_path, folder, file))
            if parent_dat is None:
                parent_dat = dat
            else:
                parent_dat = pd.concat([parent_dat, dat], ignore_index=True)

##  Predicting if a storm will be damaging

In [ ]:
prepped_data = prep_data(parent_dat, drop_zeros=False).dropna()

prepped_data

In [ ]:
X = prepped_data.drop(columns=['DAMAGE_PROPERTY'])
y = prepped_data['DAMAGE_PROPERTY']

# change y to be 1 if damage > 0 and 0 if damage == 0
y = (prepped_data['DAMAGE_PROPERTY'] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.25, random_state=8)

In [ ]:
from xgboost_classifier import *
xgclass = tune_log_model(X_train, X_test, y_train, y_test, seed=8, n_iter=50)

In [ ]:
from perm_importance import *

dummy_prefixes = ["EVENT_TYPE_", "MODAL_YEAR_BUILT_BIN_",
                   "COASTAL_TYPE_SHORELINE_", "COASTAL_TYPE_WATERSHED_"]

perm_imp_train, perm_imp_test = get_perm_importance(
    X_train, y_train, X_test, y_test,
    xgclass,
    dummy_prefixes=dummy_prefixes
)

In [ ]:
# plot the permutation importance for train and test sets
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

perm_imp_train.plot(kind='barh', ax=ax[0], title='Permutation Importance (Train)')
perm_imp_test.plot(kind='barh', ax=ax[1], title='Permutation Importance (Test)')

plt.tight_layout()
plt.show()

## Predicting the damage of damaging storms 

In [ ]:
from xgboost_regress import *

In [ ]:
prepped_nonZero_data = prep_data(parent_dat)

prepped_nonZero_data = prepped_nonZero_data.dropna()
    
prepped_nonZero_data

In [ ]:
X = prepped_nonZero_data.drop(columns=['log10_damage'])
y = prepped_nonZero_data['log10_damage']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=8)

xgbregress = tune_model(X_train, X_test, y_train, y_test, n_iter=50)

In [ ]:
from perm_importance import *
perm_imp_train, perm_imp_test = get_perm_importance(X_train, y_train, X_test, y_test, xgbregress)

In [ ]:
# plot the permutation importance for the test set and train set side by side
fig, (ax1, ax2) = plt.subplots(1,2 , figsize=(12, 8))

perm_imp_train.plot.barh(ax=ax1)
ax1.set_title("Feature importances using permutation on training set")
ax1.set_xlabel("Mean decrease in RMSE")

perm_imp_test.plot.barh(ax=ax2)
ax2.set_title("Feature importances using permutation on test set")
ax2.set_xlabel("Mean decrease in RMSE")

plt.tight_layout()
plt.show()